# Milan mobile traffic forecasting: full pipeline

Runs the whole project on the real Telecom Italia data, in the same order as `run_all.sh`: ingestion, EDA, time-series analysis, baselines, tuning rounds, final test runs, comparison and the report tables.

I use this because my laptop cannot hold the 20.8 GB of raw files. Kaggle gives about 30 GB of RAM and a free GPU for the LSTM rounds.

On Kaggle, add a dataset with the 62 `sms-call-internet-mi-YYYY-MM-DD.txt` files (Add data > search "Telecom Italia" or "sms-call-internet-mi"). The next cell searches `/kaggle/input` for those filenames, so the folder name does not matter. Check you have all 62 days first, because several public copies only cover the first week. If you want the LSTM rounds to go faster, turn on a GPU under Settings > Accelerator.

On Colab, download the files once from https://doi.org/10.7910/DVN/EGZHFV (there is a guestbook to fill in) into Drive, mount it, and point `RAW_DIR` at that folder.

## The problem

Given hourly Internet activity for one grid cell and a forecast origin t, predict the activity at t+1, t+6 and t+24 hours using only data up to and including t. Each horizon gets its own model (the direct strategy), so a 24-hour error is not 24 one-hour errors compounded and the three horizons stay independent measurements.

The data is the Telecom Italia Big Data Challenge release: SMS, call and Internet activity on a 100x100 grid over Milan at 10-minute resolution, for the 62 days from 1 Nov 2013 to 1 Jan 2014, 20.8 GB of TSV. I forecast Internet activity only, which is 82 % of all recorded activity, on 7 cells picked in the EDA stage below - never on the citywide total, whose smoothness flatters every model.

The split is chronological and identical for every model: train 1 Nov - 14 Dec, validation 15 - 21 Dec, test 22 Dec - 1 Jan. Forecasts are issued from every hour of the evaluation window, 241 per model, cell and horizon. The test window covers Christmas and New Year on purpose, because forecasting the holiday fortnight from six ordinary weeks is the situation an operator actually faces once a year. It is not touched until the final runs in Stage 6.

The headline metric is MASE: mean absolute error divided by the in-sample error of the 24-hour seasonal naive. Below 1 means a method beats that baseline on the training scale; above 1 means it loses to it. It is scale-free, which matters because the selected cells differ in volume by more than an order of magnitude and raw errors would not be comparable across them.

The question the whole pipeline exists to answer: how do a statistical model (SARIMAX), a gradient-boosted tree model (LightGBM) and a recurrent network (LSTM/GRU) compare across cells and across horizons, and does any of them beat a properly specified naive benchmark?

In [ ]:
import os, sys, subprocess, pathlib

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

def find_raw_dir(root):
    """Return the folder that actually contains sms-call-internet-mi-*.txt files."""
    root = pathlib.Path(root)
    hits = sorted(root.rglob("sms-call-internet-mi-*.txt"))
    if not hits:
        print("nothing matching sms-call-internet-mi-YYYY-MM-DD.txt under", root)
        if root.exists():
            print("what is there:")
            for p in sorted(root.rglob("*"))[:40]:
                print(" ", p)
        return str(root)
    raw_dir = str(hits[0].parent)
    print(f"found {len(hits)} daily files in {raw_dir}")
    return raw_dir

if ON_KAGGLE:
    # don't assume the dataset slug; search whatever you attached under /kaggle/input
    RAW_DIR = find_raw_dir("/kaggle/input")
    WORK = "/kaggle/working"
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW_DIR = find_raw_dir("/content/drive/MyDrive/milan_raw")
    WORK = "/content"
else:
    RAW_DIR = find_raw_dir("data/raw")
    WORK = "."
print("platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local", "| raw dir:", RAW_DIR)

In [ ]:
%cd {WORK}
if not pathlib.Path("Time-Series-Forecasting").exists():
    !git clone -q https://github.com/Samkwizera/Time-Series-Forecasting.git
%cd Time-Series-Forecasting
!pip install -q -r requirements.txt
!pip install -q -e .

The default config looks for the raw files in `data/raw`, which is not where Kaggle puts them. So I copy the config and swap in `RAW_DIR`. Every other path stays relative, so the Parquet intermediates land in the writable working directory.

The next cell counts the 62 daily files locally. It does not call Harvard Dataverse, because that API keeps returning 403 from Kaggle.

In [ ]:
import yaml
cfg = yaml.safe_load(open("config/default.yaml"))
cfg["paths"]["raw_dir"] = RAW_DIR
yaml.safe_dump(cfg, open("config/kaggle.yaml", "w"), sort_keys=False)
os.environ["MILAN_CONFIG"] = "config/kaggle.yaml"

def run(script, *args):
    """Run a pipeline script and stop the notebook if it fails."""
    cmd = [sys.executable, script, "--config", "config/kaggle.yaml", *args]
    print(">", " ".join(cmd))
    subprocess.run(cmd, check=True)

run("scripts/00_download.py", "--verify")

## Stage 1: ingestion

This is the slow part. On the full 20.8 GB it takes about an hour. Wait until it prints the `hourly_*.parquet` paths.

Nothing below works without it. Skip it or stop it early and EDA will go looking for `data/processed/citywide_10min.parquet` and fail.

In [ ]:
run("scripts/01_ingest.py")
from pathlib import Path
import pandas as pd
city = Path("data/processed/citywide_10min.parquet")
assert city.exists(), (
    f"ingest did not write {city}. Scroll up: either the raw files were missing "
    "or the ingest cell was interrupted. Re-run this cell and wait for it to finish."
)
pd.read_csv("reports/tables/memory_log.csv").tail(8)

### What ingestion actually does

Loading all 62 days naively with default pandas dtypes would take about 16.7 GB, several times the RAM a free Kaggle or Colab session gives you. Three decisions close that gap, and the memory log above is the measurement rather than the claim.

Each file is scanned lazily and rows are summed over `country_code` *during* the scan. Every (cell, interval) pair appears once per country code, so collapsing that dimension removes roughly 90 % of all rows before a frame exists in memory; of the three decisions this is the one that makes the pipeline feasible at all. Activities are then downcast to float32 and cell ids to uint16, and one zstd-compressed Parquet file per day is written, from which a wide hourly matrix per activity is built. The 20.8 GB of raw TSV is read exactly once.

Two preprocessing details matter downstream. Timestamps are converted from UTC to Europe/Rome once, here, so that "8 a.m." means the same thing in every later stage and nothing downstream has to reason about time zones. And because the daily files are cut at UTC midnight while the analysis runs in local time, the same local hour appears in two files - so the hourly matrix is re-aggregated by timestamp after concatenation instead of simply concatenated, which would silently split those hours. Missing (cell, interval) pairs mean no activity and are filled with 0.

Peak RSS in the per-day loop was 1,551 MB. One measurement runs against my own design: the later citywide 10-minute aggregation peaks at 5,244 MB, the highest point of the whole pipeline, because that stage builds a fine-grained frame that aggregating early does not protect. It fits inside Kaggle, but it is the stage I would restructure first.

## Stages 2-3: exploratory and time-series analysis

EDA makes the citywide plots, the spatial maps, and the k-means clustering that picks which cells we forecast (`selected_cells.json`). TSA then runs the stationarity tests, STL/MSTL and ACF/PACF on those cells.

I only show a few figures here. The rest are in `reports/figures/`.

In [ ]:
run("scripts/02_eda.py")
run("scripts/03_tsa.py")
from IPython.display import Image, display
for f in ["eda_citywide_hourly", "eda_daily_profiles", "eda_spatial_internet", "eda_clusters_internet", "tsa_acf_citywide"]:
    display(Image(f"reports/figures/{f}.png", width=900))

### What the figures show

Two cycles dominate: a daily one and a weaker weekly one with lower weekends, and the last week of December departs from both. The daily profiles split that by day type. Weekdays ramp from 6 a.m. and peak in the evening, weekends have no morning ramp, and public holidays look like Sundays. SMS and calls peak earlier than Internet, which is why I forecast Internet alone (82 % of all recorded activity) rather than pooling the activities into one target.

Activity is very unevenly spread: the busiest 1 % of cells carry 12 % of Internet traffic and the busiest 10 % carry 50 % (Gini 0.62). The maps also move through the day, with the centre dominating office hours and the residential ring rising by 9 p.m. So evaluation is per cell and never on the citywide aggregate, whose smoothness flatters every model.

The clustering is what picks the cells to forecast. The silhouette score actually favours k=2; I keep k=4 because it separates two business-shaped from two mixed/suburban-shaped profiles that I want to compare. That is an interpretability choice made against the metric, not an optimum.

The ACF is where the SARIMA specification comes from. Citywide, rho(1)=0.954 and rho(24)=0.927, rho(168)=0.794 is still substantial, and rho(6)=0.120 is nearly zero. STL puts 81 % of the log-series variance in the daily component and MSTL a further 9 % in the weekly one. A seasonal difference at lag 24 makes the series stationary under both ADF and KPSS, and a further first difference is not needed.

That fixes most of the modelling choices below: d=0 and D=1, a log1p transform, tree lags of 1-24 plus daily steps out to 168, one week as the candidate input window for the recurrent models, and calendar features for the *target* hour and not only the origin. The near-zero rho(6) also predicts, before anything is fitted, that h=6 is where persistence-style forecasts will fail.

## Stages 4-5: baselines and tuning rounds

The three naive baselines run first, on both splits, so every later number has something to be compared against.

Each tuning round lives in `experiments/tuning_plan.yaml` with a `why` field I fill in before running it, based on what the last round or the ACF suggested. Every run appends a row to `experiments/experiment_log.md`, so the log is the actual trail of what I tried and not a summary I wrote at the end.

SARIMA and LSTM rounds are manual. For LightGBM the manual rounds ablate the feature groups first, then 20 Optuna trials handle the capacity parameters, where I had no strong prior. If a result suggests something new, add a round to the YAML and rerun `05_tune.py --rounds <id>`.

### The three families, and why these three

They span different assumptions about what generates the series, which is what makes their relative performance interpretable instead of a leaderboard.

**Baselines** - naive, 24-hour seasonal naive, 168-hour seasonal naive. The 24-hour one is also the scaling series for MASE. These set the bar a learned model has to clear to have earned its complexity, which is why they run first.

**SARIMAX** - a linear seasonal process, (p,0,q)(P,1,Q) with period 24, fitted per cell. A seasonal period of 168 would make the state vector far too large, so the weekly cycle is offered as optional Fourier regressors instead and validation decides whether to keep them. Parameters are estimated once on the fitting window and forecasts are read with `dynamic=True`, so they are genuine multi-step forecasts and not filtered one-step updates.

**LightGBM** - nonlinear interactions over engineered lags, one regressor per horizon, trained jointly across the selected cells with a categorical cell id so the trees share hour-of-day structure while still splitting on cell where profiles differ. The objective is L1: it targets the conditional median, is robust to the heavy right tail of peak hours, and is the same loss that MAE and MASE measure.

**LSTM / GRU** - a learned representation of the recent sequence, reading 24 or 168 hours and emitting all three horizons from the final hidden state together with a learned cell embedding.

The tree and the network receive the same calendar information, each in the form it can use, so what is being compared is inductive bias and not feature access. Spatial CNNs are deliberately left out: judging neighbour information fairly needs a different input and sampling design, and mixing that into a model-family comparison would make both questions harder to read.

In [ ]:
run("scripts/04_train.py", "--model", "baselines", "--part", "val")
run("scripts/04_train.py", "--model", "baselines", "--part", "test")
run("scripts/05_tune.py", "--model", "sarima")
run("scripts/05_tune.py", "--model", "lightgbm", "--optuna", "20")
run("scripts/05_tune.py", "--model", "lstm")
print(open("experiments/experiment_log.md").read())

### Reading the log

SARIMAX: adding MA terms at lags 1 and 24 (s2) cut validation MASE from 0.717 to 0.659, and nothing after that improved on it - weekly Fourier regressors 0.666, a holiday dummy 0.669, a larger specification 0.769. The Fourier terms are the interesting failure, because rho(168)=0.794 is a real feature of the ACF and they still added nothing. The weekly cycle appears to be already covered by the daily structure and the recent lags.

LightGBM: calendar features of the target hour gave the best run of the whole study (g3, 0.604), which is the point the EDA predicted - a model has to know what kind of day it is forecasting into. Rolling statistics made it worse (0.664). The best of 20 Optuna trials reached 0.633 and still did not beat g3, but the search held `use_rolling=True` fixed, so what it shows is that no capacity setting rescues the rolling-feature variant. It does not show that g3 is a tuned optimum.

Recurrent: a week of context did not help (l2 at 0.747 against l1 at 0.721 with a day of context) until the network was made wider (l3, 0.714), and the GRU reached 0.711. Those last two are 0.003 apart on a single seed, smaller than the run-to-run variation I would expect from initialisation alone. I took the GRU on the tie-break that it has fewer parameters at equal width, not because it was shown to be better.

So several changes with sound time-series reasoning behind them failed to generalise even to the validation week. That is worth noticing before the test runs, because it is the first hint of what follows.

## Stage 6: final test runs and comparison

For each model I take the config with the lowest validation MASE and run it once on the test split, 22 Dec to 1 Jan. That window covers the holidays on purpose. The test period is not touched before this point.

`06_compare.py` then scores everything against the baselines, runs the Diebold-Mariano tests and writes the failure-analysis figures.

In [ ]:
# same selection rule as run_all.sh, written out here so the chosen runs are visible in the output
import json, glob
def best(model):
    runs = [json.load(open(p)) for p in glob.glob(f"experiments/runs/{model}/*_val.json")]
    r = min(runs, key=lambda r: r["metrics"]["mase"])
    return r["run_id"].removesuffix("_val"), r["params"]
runs = {m: best(m) for m in ["sarima", "lightgbm", "lstm"]}
for m, (rid, params) in runs.items():
    print(m, rid, {k: v for k, v in params.items() if k in ("order","seasonal_order","fourier_k","lags","num_leaves","learning_rate","cell","hidden_size","num_layers","input_window")})
for m, (rid, params) in runs.items():
    cell = params.get("cell", m) if m == "lstm" else m
    run("scripts/04_train.py", "--model", cell, "--part", "test", "--run", rid,
        "--params", json.dumps(params), "--note", "final test run of best validation config")
lstm_cell = runs["lstm"][1].get("cell", "lstm")
run("scripts/06_compare.py", "--runs", f"sarima={runs['sarima'][0]}",
    f"lightgbm={runs['lightgbm'][0]}", f"{lstm_cell}={runs['lstm'][0]}")
pd.read_csv("reports/tables/results_summary.csv")

In [ ]:
for f in ["results_mase_by_horizon_and_cell", "results_forecasts_h1", "failure_daily_error", "failure_hour_daytype"]:
    display(Image(f"reports/figures/{f}.png", width=900))

### What the comparison shows

The 24-hour seasonal naive is the strongest method overall, at MASE 0.863. SARIMAX (1.427) and LightGBM (1.432) are both above 1, which means worse than that baseline on the training scale, and the GRU is far behind at 2.223. The Diebold-Mariano tests agree with the averages rather than rescuing the models: against the seasonal naive, SARIMAX is significantly better in 7 of 21 cell-horizon comparisons and worse in 10, LightGBM better in 2 and worse in 14, the GRU better in 2 and worse in 15.

The horizon separates the methods far more sharply than the model family does. SARIMAX is the single best method in the study at one hour (0.399) and then degrades steeply to 2.341 at 24 hours; LightGBM degrades much less and overtakes it there (1.568). Overall SARIMAX and LightGBM differ by 0.005 MASE, far less than the spread across cells and horizons, so I treat them as indistinguishable in aggregate and do not rank them.

The failure figures say where the error sits. Daily error peaks on 25-26 December, and relative error is highest on holidays for every learned model (LightGBM 0.58 on holidays against 0.48 on weekdays, the GRU 0.92 against 0.76) while the naive methods are the least disturbed by them. The test window was chosen to cover Christmas on purpose, so this is the stress test working as intended and not a surprise.

One caveat on all of it: a single test window, seven cells and one seed. The DM tests are the only uncertainty estimate here.

## Stage 7: report assets

No number in the report is typed by hand. `07_report_assets.py` turns the CSVs into LaTeX tables and `\newcommand` macros under `report/generated/`.

The last cell zips `reports/`, `experiments/` and those generated files. Kaggle has no LaTeX, so I unpack them in my local clone and build the PDF there.

In [ ]:
run("scripts/07_report_assets.py")
!zip -qr results_bundle.zip reports experiments report/generated
print("download results_bundle.zip, unpack it in the local clone, then run report/build.sh")

## Conclusion

The 24-hour seasonal naive is the best method overall, at MASE 0.863, and no learned model beats it beyond one hour ahead. That is a negative result and it is the real one rather than a bug: a holiday fortnight is mainly a shift in level, and re-anchoring on a recent observation absorbs that shift without having to estimate it, which is why the baseline's error barely grows with horizon while SARIMAX's degrades from 0.399 at h=1 to 2.341 at h=24.

SARIMAX (1.427) and LightGBM (1.432) are indistinguishable in aggregate and are better read by horizon - SARIMAX is the strongest method in the study at one hour, LightGBM the stronger of the two at 24. The GRU (2.223) does not justify its complexity on six weeks of fitting data. Ranking varies systematically with horizon but not demonstrably with cell profile, so on that half of the research question the evidence does not support a positive claim either way.

What I would do next, in order. Give the models the benchmark's own advantage explicitly, by forecasting the residual from the last observed same-hour value, and test whether the long-horizon collapse disappears. Train on all 10,000 cells with cell-type embeddings, which would give the sequence model the training volume it clearly lacked. And use a longer dataset covering several holiday periods, since that is the only way to estimate holiday effects instead of merely flagging them.

Limits to hold in mind when reading any of the above: seven cells, one activity, one seed, a single seven-day validation window that contains no holiday, and a deliberately hard test period. That supports claims about large differences between model families, not small differences between configurations within one.